# Maritime Edge LLM Evaluation

This notebook evaluates Gemma 4 E4B on maritime tactical scenarios.

**Steps:**
1. Upload `train_data.jsonl` (use file browser on left)
2. Run all cells
3. Results saved to `results.json`

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft datasets

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
import json
import time
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Load model (4-bit quantized to fit in T4 GPU)
MODEL_ID = "google/gemma-3-4b-it"  # Using Gemma 3 4B as Gemma 4 E4B may not be on HF yet

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print("Model loaded!")

In [ ]:
# Load scenarios
scenarios = []
with open("train_data.jsonl") as f:
    for line in f:
        scenarios.append(json.loads(line))

print(f"Loaded {len(scenarios)} scenarios")

In [ ]:
# Prompt template
SYSTEM_PROMPT = """You are a tactical decision system for an autonomous maritime drone.

Given sensor data about nearby vessels, output a JSON decision:
{"threat_level": "none|low|medium|high|critical", "action": "continue|monitor|evade|alert|abort", "reasoning": "brief explanation", "confidence": 0.0-1.0}

Output ONLY valid JSON, no other text."""

def format_scenario(scenario):
    s = scenario
    vessels = []
    for i, v in enumerate(s["vessels"], 1):
        ais = "AIS" if v["ais_active"] else "NO AIS"
        vessels.append(f"  C{i}: {v['vessel_type']} @ {v['bearing']:.0f}°/{v['distance']:.1f}nm, {v['speed']:.0f}kn, {ais}")
    
    return f"""Mission: {s['mission_type']}
Own ship: {s['own_heading']:.0f}°/{s['own_speed']:.0f}kn
Conditions: {s['weather']}, vis={s['visibility']}, {s['time_of_day']}, comms={s['comms_status']}
Contacts:
{chr(10).join(vessels)}

Decision:"""

# Test formatting
print(format_scenario(scenarios[0]["scenario"]))

In [ ]:
def predict(scenario, max_new_tokens=150):
    """Run inference on a single scenario."""
    prompt = format_scenario(scenario)
    
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, 
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# Test on first scenario
print("Testing inference...")
result = predict(scenarios[0]["scenario"])
print(f"Response: {result}")

In [ ]:
def parse_response(response):
    """Extract JSON from model response."""
    try:
        # Try direct parse
        return json.loads(response)
    except:
        pass
    
    # Try to find JSON in response
    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response[start:end])
    except:
        pass
    
    return None

# Test parsing
parsed = parse_response(result)
print(f"Parsed: {parsed}")

In [ ]:
# Run evaluation on subset (full 300 takes ~30 min on T4)
N_EVAL = 50  # Change to 300 for full eval

results = []
correct_threat = 0
correct_action = 0
correct_full = 0
parse_errors = 0

print(f"Evaluating {N_EVAL} scenarios...")
start_time = time.time()

for i, item in enumerate(scenarios[:N_EVAL]):
    scenario = item["scenario"]
    ground_truth = item["decision"]
    
    # Get prediction
    response = predict(scenario)
    parsed = parse_response(response)
    
    if parsed is None:
        parse_errors += 1
        results.append({"id": scenario["id"], "error": "parse_error", "response": response})
        continue
    
    # Compare
    pred_threat = parsed.get("threat_level", "").lower()
    pred_action = parsed.get("action", "").lower()
    gt_threat = ground_truth["threat_level"].lower()
    gt_action = ground_truth["action"].lower()
    
    threat_match = pred_threat == gt_threat
    action_match = pred_action == gt_action
    
    if threat_match:
        correct_threat += 1
    if action_match:
        correct_action += 1
    if threat_match and action_match:
        correct_full += 1
    
    results.append({
        "id": scenario["id"],
        "pred_threat": pred_threat,
        "pred_action": pred_action,
        "gt_threat": gt_threat,
        "gt_action": gt_action,
        "threat_match": threat_match,
        "action_match": action_match,
        "reasoning": parsed.get("reasoning", ""),
    })
    
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"  {i+1}/{N_EVAL} done ({elapsed:.1f}s)")

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.1f}s ({elapsed/N_EVAL:.2f}s per scenario)")

In [ ]:
# Print results
valid = N_EVAL - parse_errors

print("\n" + "="*50)
print("RESULTS: Raw Gemma 4B (no fine-tuning)")
print("="*50)
print(f"\nScenarios evaluated: {N_EVAL}")
print(f"Parse errors: {parse_errors} ({parse_errors/N_EVAL*100:.1f}%)")
print(f"\nAccuracy (on {valid} valid responses):")
print(f"  Threat Level: {correct_threat/valid*100:.1f}%")
print(f"  Action:       {correct_action/valid*100:.1f}%")
print(f"  Full (both):  {correct_full/valid*100:.1f}%")

# Per-class breakdown
print("\nPer-class accuracy:")
for level in ["none", "low", "medium", "high", "critical"]:
    matches = [r for r in results if "gt_threat" in r and r["gt_threat"] == level]
    if matches:
        acc = sum(1 for r in matches if r["threat_match"]) / len(matches)
        print(f"  {level}: {acc*100:.1f}% ({len(matches)} samples)")

In [ ]:
# Save results
with open("results_raw_gemma.json", "w") as f:
    json.dump({
        "model": MODEL_ID,
        "n_eval": N_EVAL,
        "parse_errors": parse_errors,
        "accuracy": {
            "threat": correct_threat / valid,
            "action": correct_action / valid,
            "full": correct_full / valid,
        },
        "results": results,
    }, f, indent=2)

print("Results saved to results_raw_gemma.json")
print("\nDownload this file and copy the accuracy numbers to your README.")

In [ ]:
# Show some example predictions
print("\nExample predictions:")
for r in results[:5]:
    if "error" in r:
        print(f"  {r['id']}: PARSE ERROR")
    else:
        match = "✓" if r["threat_match"] and r["action_match"] else "✗"
        print(f"  {r['id']}: pred={r['pred_threat']}/{r['pred_action']} gt={r['gt_threat']}/{r['gt_action']} {match}")